# Digital Twin Diabetes Hardlopen

In [1]:
print('diabetes')


diabetes


Equations and ODEs - SSA 3

In [ ]:
# Distribution volumes of glucose and insulin; distributie volume = theoretisch maatvoering in de farmacokinetiek die aangeeft hoe ver een geneesmiddel zich verspreid door het lichaam.; laat zien hoeveel stof er in de weefsels zit tov in het bloed
# VI = 0.12*BW      # volume insuline
# VG = 0.16*BW      # volume glucose

# Plasma insulin concentration
# dI_dt = UI(t)/VI - ke*I(t)

# Insulin sensitivity of disposal
# SID = kb2/ka2

# Two-compartment carbohydrate absorption
# dD1_dt = AG*d - D1/tauD
# dD2_dt = D1/tauD - D2/tauD

# Glucose appearance rate
# UG = D2 / tauD

# Two-compartment insulin absorption
# dS_dt = u - S1/tauS
# dS2_dt = S1/tauS - S2/tauS

# Insulin appearance rate
# UI = S2/tauS

# Two-compartment glucose kinetics
# dQ1_dt = -(Fc010/(VG*G)+x1)*Q1 + k12*Q2 - FR + UG + EGP0 - EGP0*x3
# dQ2_dt = x1*Q1 - k12*Q2*y*G - x2*Q2*y*G

In [ ]:
# Basal values

Gluc_base = 120                             # basal glucose level [mg/dl]                      # weight [kg]
Ins_base = 10                               # basal insulin level [muU/ml]
BW = 60                                     # Body Weight [kg]


In [ ]:

"""" Lineaire stochastische differentiaalvergelijk (SDE) voor de bloedglucose G(t),
vaak in de vorm van een Ornstein-Uhlenbeck-proces met extra termen voor voeding en insuline.

"""
import numpy as np

def predict_glucose_24h(
    age,              # jaren
    sex,              # 'M' of 'F'
    height_cm,        # cm
    weight_kg,        # kg
    G0,               # huidige glucose [mmol/L]
    trend_arrow,      # bijv. -2,-1,0,1,2 (sterkte richting)
    dt_min=5,         # tijdstap in minuten
    horizon_h=24,     # voorspelhorizon in uren
    G_target=6.0,     # gewenste evenwicht-glucose [mmol/L], kan ook persoon-specifiek
):
    """ Discrete glucose-voorspelling voor type 1 diabetes, gebaseerd op een vereenvoudigd
    fysiologisch model

    Inputs:
    - age, sex, height_cm, weight_kg: demografie
    - G0: huidige glucose [mmol/L]
    - trend_arrow: richting/snelheid van verandering, bijv.:
        -2 = snel dalend
        -1 = dalend
         0 = stabiel
        +1 = stijgend
        +2 = snel stijgend
    - dt_min: tijdstap (default 5 minuten, zoals CGM)
    - horizon_h: hoe ver vooruit (default 24 uur)
    - G_target: persoonlijke streefglucose (default 6.0 mmol/L)
    
    Output:
    - t_min: array met tijden [minuten]
    - G_pred: voorspelde glucose [mmol/L]
    - G_lower, G_upper: 95% betrouwbaarheidsband (ongeveer)
    """
# 1. Basisparameters schalen met persoonskenmerken

# BMI als eenvoudige proxy voor lichaamssamenstelling
height_m = height_cm /100.0
bmi = weight_kg/(height_m**2)

# Insulinegevoeligheid (ruwe schatting, eenheid: mmol/L per 'trend-eenheid')
# Zwaarder -> vaak lagere gevoeligheid; jonger -> vaak hogere gevoeligheid
# Dit is een heuristiek, geen klinisch gevalideerde formule
base_SI = 0.05 # basale schaal
SI = base_SI*(70.0/weight_kg)*(30.0/max(age,10))    # grof geschaald

# Terugtrekkracht gamma: hoe snel glucose naar evenwicht terugkeert
# Eenheid: 1/min. Typisch tijdconstante ong. 30-60 min -> gamma ong 1/30 -1/60
base_gamma = 1.0/45.0   # 1/min
# iets aanpassen voor BMI/gewicht: zwaardere personen hebben vaak tragere dynamiek
gamma = base_gamma*(25.0/max(bmi, 15.0))


test